# 6 WorkFlow  Gerencial, futuro=JULIO

### 6.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán enriquecer

#### 6.2  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab
*   Bajar el **dataset_historico** al Google Drive y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dm"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dm"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"
descargar  "gerencial_competencia_2026.csv.gz"

cp: cannot stat '/content/buckets/b1/kaggle/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


## 6.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Nov 02 12:20:51 AM 2026"

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671310,35.9,1473341,78.7,1473341,78.7
Vcells,1242682,9.5,8388608,64.0,1978743,15.1


In [ ]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘R.utils’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘R.oo’, ‘R.methodsS3’


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach,

#### Parametros
Si es gerente, no cambie nada
<br>Si es Analista, cambie el nombre del dataset

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 484259   # la usa el grid search; no cambiar para reproducir
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

# =====================================================================
# FASE 2 (CONFIRMACION): dejar activo UN solo bloque (sin #) y comentar el resto.
# En Colab: seleccionar las lineas del bloque y apretar Ctrl + /
# Los meses que no figuran en pesos_mes valen 1.
# =====================================================================

# --- F2 W0 control: todos los meses con peso 1 ---
PARAM$experimento <- 9200
PARAM$variante <- "F2_W0_baseline"
PARAM$pesos_mes <- numeric(0)

# --- F2 W3: decaimiento lineal de 0.20 a 1 ---
# PARAM$experimento <- 9230
# PARAM$variante <- "F2_W3_decaimiento_lineal_0.2_a_1"
# PARAM$pesos_mes <- c(
#   "202005" = 0.20, "202006" = 0.27, "202007" = 0.35, "202008" = 0.42,
#   "202009" = 0.49, "202010" = 0.56, "202011" = 0.64, "202012" = 0.71,
#   "202101" = 0.78, "202102" = 0.85, "202103" = 0.93, "202104" = 1.00
# )

# --- F2 W6: escalones ascendentes 0.25 / 0.5 / 0.75 / 1 ---
# PARAM$experimento <- 9260
# PARAM$variante <- "F2_W6_escalones_ascendentes_0.25_0.5_0.75_1"
# PARAM$pesos_mes <- c(
#   "202005" = 0.25, "202006" = 0.25, "202007" = 0.25,
#   "202008" = 0.50, "202009" = 0.50, "202010" = 0.50,
#   "202011" = 0.75, "202012" = 0.75, "202101" = 0.75,
#   "202102" = 1.00, "202103" = 1.00, "202104" = 1.00
# )

# --- F2 W7: escalones descendentes 1 / 0.75 / 0.5 / 0.25 (control negativo) ---
# PARAM$experimento <- 9270
# PARAM$variante <- "F2_W7_escalones_descendentes_1_0.75_0.5_0.25"
# PARAM$pesos_mes <- c(
#   "202005" = 1.00, "202006" = 1.00, "202007" = 1.00,
#   "202008" = 0.75, "202009" = 0.75, "202010" = 0.75,
#   "202011" = 0.50, "202012" = 0.50, "202101" = 0.50,
#   "202102" = 0.25, "202103" = 0.25, "202104" = 0.25
# )

# --- F2 W10: meses del ASPO (may-20 a oct-20) en 0.5 ---
# PARAM$experimento <- 9300
# PARAM$variante <- "F2_W10_ASPO_0.5_resto_1"
# PARAM$pesos_mes <- c(
#   "202005" = 0.5, "202006" = 0.5, "202007" = 0.5,
#   "202008" = 0.5, "202009" = 0.5, "202010" = 0.5
# )

# =====================================================================
# 30 semillas nuevas (numeros primos); se pueden reemplazar por las propias
PARAM$semillas <- c(
  133811, 159763, 175403, 208309, 249589, 254039, 254279, 292793, 325421, 393517,
  471187, 475889, 579653, 581617, 596573, 609221, 670597, 700591, 705317, 718457,
  781511, 830591, 840061, 840179, 889657, 917333, 921259, 930157, 978067, 993611
)

# hora de inicio, para calcular los minutos de la corrida
t0 <- Sys.time()

#### Carpeta del Experimento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 6.3.1   Preprocesamiento del dataset

#### 6.3.1.1  DT incorporar dataset

In [ ]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

ERROR: Error in fread(paste0("/content/datasets/", PARAM$dataset)): could not find function "fread"


#### 6.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [ ]:
dataset[ foto_mes==202006, internet:=NA]
dataset[ foto_mes==202006, mrentabilidad:=NA]
dataset[ foto_mes==202006, mrentabilidad_annual:=NA]
dataset[ foto_mes==202006, mcomisiones:=NA]
dataset[ foto_mes==202006, mactivos_margen:=NA]
dataset[ foto_mes==202006, mpasivos_margen:=NA]
dataset[ foto_mes==202006, mcuentas_saldo:=NA]
dataset[ foto_mes==202006, ctarjeta_visa_transacciones:=NA]
dataset[ foto_mes==202006, mtarjeta_visa_consumo:=NA]
dataset[ foto_mes==202006, mtarjeta_master_consumo:=NA]
dataset[ foto_mes==202006, ccallcenter_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]

#### 6.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, quizas ajustando por IPC ...
<br>Esta parte podrá ser abordada por todos los Analistas y también la Gerenciapero se decide pedagogicamente no incluirla en esta primer version para reducir la carga cognitiva

In [ ]:
# sin codigo en esta primera version del workflow

#### 6.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [ ]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


In [ ]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

[1] "numero_de_cliente"           "foto_mes"                   
 [3] "internet"                    "cliente_edad"               
 [5] "cliente_antiguedad"          "mrentabilidad"              
 [7] "mrentabilidad_annual"        "mcomisiones"                
 [9] "mactivos_margen"             "mpasivos_margen"            
[11] "cproductos"                  "mcuenta_corriente"          
[13] "mcaja_ahorro"                "cdescubierto_preacordado"   
[15] "mcuentas_saldo"              "ctarjeta_visa_transacciones"
[17] "mtarjeta_visa_consumo"       "mtarjeta_master_consumo"    
[19] "mprestamos_personales"       "cpayroll_trx"               
[21] "mpayroll"                    "ccomisiones_mantenimiento"  
[23] "ccallcenter_transacciones"   "chomebanking_transacciones" 
[25] "ctrx_quarter"                "Master_status"              
[27] "Master_fechaalta"            "Master_mpagominimo"         
[29] "Visa_status"                 "Visa_fechaalta"             
[31] "Visa_mpagominimo"            "clase_ternaria"             
[33] "kmes"                        "mpayroll_sobre_edad"

#### 6.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

Esto se mostrará unicamente a la *modalidad Analista Sr*

In [ ]:
# No se implementa Feature Engineering a partir de Random Forest

#### 6.3.1.5  FEhist Feature Engineering historico

El Feature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [ ]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


Verificacion de los campos recien creados

In [ ]:
ncol(dataset)
colnames(dataset)

[1] 158

[1] "numero_de_cliente"                  "foto_mes"                          
  [3] "internet"                           "cliente_edad"                      
  [5] "cliente_antiguedad"                 "mrentabilidad"                     
  [7] "mrentabilidad_annual"               "mcomisiones"                       
  [9] "mactivos_margen"                    "mpasivos_margen"                   
 [11] "cproductos"                         "mcuenta_corriente"                 
 [13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
 [15] "mcuentas_saldo"                     "ctarjeta_visa_transacciones"       
 [17] "mtarjeta_visa_consumo"              "mtarjeta_master_consumo"           
 [19] "mprestamos_personales"              "cpayroll_trx"                      
 [21] "mpayroll"                           "ccomisiones_mantenimiento"         
 [23] "ccallcenter_transacciones"          "chomebanking_transacciones"        
 [25] "ctrx_quarter"                       "Master_status"                     
 [27] "Master_fechaalta"                   "Master_mpagominimo"                
 [29] "Visa_status"                        "Visa_fechaalta"                    
 [31] "Visa_mpagominimo"                   "clase_ternaria"                    
 [33] "kmes"                               "mpayroll_sobre_edad"               
 [35] "internet_lag1"                      "cliente_edad_lag1"                 
 [37] "cliente_antiguedad_lag1"            "mrentabilidad_lag1"                
 [39] "mrentabilidad_annual_lag1"          "mcomisiones_lag1"                  
 [41] "mactivos_margen_lag1"               "mpasivos_margen_lag1"              
 [43] "cproductos_lag1"                    "mcuenta_corriente_lag1"            
 [45] "mcaja_ahorro_lag1"                  "cdescubierto_preacordado_lag1"     
 [47] "mcuentas_saldo_lag1"                "ctarjeta_visa_transacciones_lag1"  
 [49] "mtarjeta_visa_consumo_lag1"         "mtarjeta_master_consumo_lag1"      
 [51] "mprestamos_personales_lag1"         "cpayroll_trx_lag1"                 
 [53] "mpayroll_lag1"                      "ccomisiones_mantenimiento_lag1"    
 [55] "ccallcenter_transacciones_lag1"     "chomebanking_transacciones_lag1"   
 [57] "ctrx_quarter_lag1"                  "Master_status_lag1"                
 [59] "Master_fechaalta_lag1"              "Master_mpagominimo_lag1"           
 [61] "Visa_status_lag1"                   "Visa_fechaalta_lag1"               
 [63] "Visa_mpagominimo_lag1"              "kmes_lag1"                         
 [65] "mpayroll_sobre_edad_lag1"           "internet_lag2"                     
 [67] "cliente_edad_lag2"                  "cliente_antiguedad_lag2"           
 [69] "mrentabilidad_lag2"                 "mrentabilidad_annual_lag2"         
 [71] "mcomisiones_lag2"                   "mactivos_margen_lag2"              
 [73] "mpasivos_margen_lag2"               "cproductos_lag2"                   
 [75] "mcuenta_corriente_lag2"             "mcaja_ahorro_lag2"                 
 [77] "cdescubierto_preacordado_lag2"      "mcuentas_saldo_lag2"               
 [79] "ctarjeta_visa_transacciones_lag2"   "mtarjeta_visa_consumo_lag2"        
 [81] "mtarjeta_master_consumo_lag2"       "mprestamos_personales_lag2"        
 [83] "cpayroll_trx_lag2"                  "mpayroll_lag2"                     
 [85] "ccomisiones_mantenimiento_lag2"     "ccallcenter_transacciones_lag2"    
 [87] "chomebanking_transacciones_lag2"    "ctrx_quarter_lag2"                 
 [89] "Master_status_lag2"                 "Master_fechaalta_lag2"             
 [91] "Master_mpagominimo_lag2"            "Visa_status_lag2"                  
 [93] "Visa_fechaalta_lag2"                "Visa_mpagominimo_lag2"             
 [95] "kmes_lag2"                          "mpayroll_sobre_edad_lag2"          
 [97] "internet_delta1"                    "internet_delta2"                   
 [99] "cliente_edad_delta1"                "cliente_edad_delta2"               
[1

#### 6.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  nni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [ ]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 6.3.2 Modelado

#### 6.3.2.1 Training Strategy

Esta etapa de Workflow de  Training Strategy esta pensada para la *Modalidad Gerencial* que posee el dataset de [202005, 202109]
<br> Si usted es un Analista, posee el periodo de [201901, 202109] y deberá experimentar en que meses le conviene experimentar

<br> A la *Modalidad Gerencial* no se le complicada la vida con el undersampling de los continua, por eso PARAM$trainingstrategy$training_pct <- 1.0
<br> Sin embargo, si usted es  *Analista SR* posee un dataset 50 veces ( filas x columnas) más grande que la *Modalidad Gerencial*  y por un tema de velocidad y experimentación más rápida puede llegar a necesitar activar el undersampling de la clase mayoritaria, a pesar de estar corriendo en Google Cloud.

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202107  obviamente completo

* final_train =  [ 202005, 202105 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202105   completo, sin undersampling
   * training = [ 202005, 202104 ]  donde se consideran el 100% de los CONTINUA

In [ ]:
PARAM$trainingstrategy$validate <- c(202105)

PARAM$trainingstrategy$training <- c(
  202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [ ]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [ ]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar","peso"))
)

La siguiente celda corre en interminables 8 minutos

In [ ]:
# preparo para que se pueda hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]

# peso de cada mes: 1 por defecto, los de PARAM$pesos_mes lo bajan
dataset[, peso := 1.0]
for( m in names(PARAM$pesos_mes) ) {
  dataset[ foto_mes == as.integer(m), peso := PARAM$pesos_mes[[m]] ]
}

if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  weight= dataset[fold_train == TRUE, peso],
  free_raw_data= TRUE
)

# verificacion: el peso de cada mes y que peso NO sea una variable del modelo
print( dataset[fold_train == TRUE, list(filas= .N, peso= mean(peso)), by= foto_mes][order(foto_mes)] )
print( "peso" %in% campos_buenos )   # tiene que decir FALSE

Loading required package: lightgbm

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘lightgbm’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: lightgbm



    foto_mes filas  peso
       <int> <int> <num>
 1:   202005 12080  1.00
 2:   202006 12302  1.00
 3:   202007 12476  1.00
 4:   202008 12583  0.75
 5:   202009 12697  0.75
 6:   202010 12828  0.75
 7:   202011 12896  0.50
 8:   202012 12957  0.50
 9:   202101 12995  0.50
10:   202102 13029  0.25
11:   202103 13109  0.25
12:   202104 13131  0.25
[1] FALSE


In [ ]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

[1] 13167

####  6.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [ ]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [ ]:
# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  niter <- modelo_train$best_iter
  # hago espacio en la memoria
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}

seteo del Grid Search

In [ ]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
tb_nueva <- CJ(
  num_leaves= c(64, 128, 256, 512),
  min_data_in_leaf= c(64, 256, 512, 1024, 2048)
)

Corrida del Grid Search,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer
<br> ATENCION, la siguiente celda demora 60 minutos en Colab
<br> lamento profundamente tal intolerable espera gerencial

In [ ]:
# registro a registro calculo la AUC
tb_nueva[, c("AUC", "num_iterations"):= Estimar_AUC_lightgbm( .SD ),
  by=1:nrow(tb_nueva) ]

Mon Nov 02 12:33:15 AM 2026  64, 64 niter 652 AUC 0.947207463874131

Mon Nov 02 12:34:34 AM 2026  64, 256 niter 339 AUC 0.947642086530975

Mon Nov 02 12:36:52 AM 2026  64, 512 niter 727 AUC 0.949366032699366

Mon Nov 02 12:38:59 AM 2026  64, 1024 niter 633 AUC 0.944358033246922

Mon Nov 02 12:43:40 AM 2026  64, 2048 niter 1555 AUC 0.946416074193852

Mon Nov 02 12:46:36 AM 2026  128, 64 niter 878 AUC 0.950331955887511

Mon Nov 02 12:48:38 AM 2026  128, 256 niter 541 AUC 0.951365468032135

Mon Nov 02 12:50:25 AM 2026  128, 512 niter 440 AUC 0.949878083211417

Mon Nov 02 12:52:28 AM 2026  128, 1024 niter 508 AUC 0.946886202441758

Mon Nov 02 12:57:51 AM 2026  128, 2048 niter 1922 AUC 0.945506189950634

Mon Nov 02 12:59:42 AM 2026  256, 64 niter 366 AUC 0.952200918867586

Mon Nov 02 01:03:01 AM 2026  256, 256 niter 800 AUC 0.949508055063611

Mon Nov 02 01:04:54 AM 2026  256, 512 niter 363 AUC 0.94752230863342

Mon Nov 02 01:07:32 AM 2026  256, 1024 niter 722 AUC 0.947005552561108

Mon Nov 

la optimizacion de hiperparámetros de tipo  Grid Search ha corrido, extraigo los mejores hiperparametros

In [ ]:
tb_nueva

fwrite( tb_nueva,
  file= "tb_grid_search_01.txt",
  sep="\t",
  append= TRUE
)

num_leaves,min_data_in_leaf,AUC,num_iterations
<dbl>,<dbl>,<dbl>,<int>
64,64,0.9472075,652
64,256,0.9476421,339
64,512,0.9493660,727
64,1024,0.9443580,633
64,2048,0.9464161,1555
128,64,0.9503320,878
128,256,0.9513655,541
128,512,0.9498781,440
128,1024,0.9468862,508


In [ ]:
setorder( tb_nueva, -AUC)  # ordeno DESCENDENTE por AUC
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC] # en la posicion 1 estan los mejores
PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_nueva[1] )
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$mejores_hiperparametros

$num_leaves
[1] 256

$min_data_in_leaf
[1] 64

$num_iterations
[1] 366

### 6.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización de hiperparámetros

In [ ]:
PARAM$trainingstrategy$final_train <- c(
  202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM, con el peso de cada mes
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  weight= dataset[fold_final_train == TRUE, peso],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

# verificacion: los pesos en Produccion tienen que ser los mismos que en Desarrollo
print( dataset[fold_final_train == TRUE, list(filas= .N, peso= mean(peso)), by= foto_mes][order(foto_mes)] )

[1] 166250

    foto_mes filas  peso
       <int> <int> <num>
 1:   202005 12080  1.00
 2:   202006 12302  1.00
 3:   202007 12476  1.00
 4:   202008 12583  0.75
 5:   202009 12697  0.75
 6:   202010 12828  0.75
 7:   202011 12896  0.50
 8:   202012 12957  0.50
 9:   202101 12995  0.50
10:   202102 13029  0.25
11:   202103 13109  0.25
12:   202104 13131  0.25
13:   202105 13167  1.00


##### Final Training Hyperparameters

In [ ]:
# uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en el grid search
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# modifyList pisa los valores repetidos en vez de duplicarlos
param_final <- modifyList(fijos, PARAM$out$lgbm$mejores_hiperparametros)

# verificacion: deberia mostrar los valores ganadores del grid
print(param_final[c("num_leaves", "min_data_in_leaf", "num_iterations")])

$num_leaves
[1] 256

$min_data_in_leaf
[1] 64

$num_iterations
[1] 366



##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [ ]:
# un modelo final por cada semilla; lo unico que cambia es la semilla del LightGBM
final_models <- list()

for( s in PARAM$semillas ) {
  param_s <- param_final
  param_s$seed <- s

  final_models[[ as.character(s) ]] <- lgb.train(
    data= dfinal_train,
    param= param_s,
    verbose= -100
  )
}

length( final_models )   # verifico la cantidad de modelos

[1] 30

In [ ]:
# grabo a disco los modelos en un formato para seres humanos ... ponele ...
#  cada semilla en su propia carpeta, asi no se pisan

for( s in PARAM$semillas ) {
  dir.create( paste0("semilla_", s), showWarnings= FALSE )
  lgb.save( final_models[[ as.character(s) ]],
    filename= paste0("semilla_", s, "/modelo.txt") )
}

In [ ]:
# ahora imprimo la importancia de variables, una por semilla

for( s in PARAM$semillas ) {
  tb_importancia <- as.data.table( lgb.importance( final_models[[ as.character(s) ]] ) )

  fwrite( tb_importancia,
    file= paste0("semilla_", s, "/impo.txt"),
    sep= "\t"
  )
}

#### Scoring

Aplico el modelo final a los datos del futuro

In [ ]:
PARAM$trainingstrategy$future <- c(202107)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

In [ ]:
# aplico cada final_model   a dfuture

mat_future <- data.matrix( dfuture[, campos_buenos, with= FALSE] )

predicciones <- list()
for( s in PARAM$semillas ) {
  predicciones[[ as.character(s) ]] <- predict( final_models[[ as.character(s) ]], mat_future )
}

##### Tabla Prediccion

In [ ]:
# una tabla de prediccion por semilla
tb_predicciones <- list()

for( s in PARAM$semillas ) {
  tb_prediccion <- dfuture[, list(numero_de_cliente)]
  tb_prediccion[, prob := predicciones[[ as.character(s) ]] ]

  # grabo las probabilidad del modelo
  #  me va a ser util para hacer Ensembles de modelos
  fwrite(tb_prediccion,
    file= paste0("semilla_", s, "/prediccion.txt"),
    sep= "\t"
  )

  tb_predicciones[[ as.character(s) ]] <- tb_prediccion
}

#### Curva de Ganancia

Genero las salidas

In [ ]:
for( s in PARAM$semillas ) {
  tb_prediccion <- tb_predicciones[[ as.character(s) ]]

  # asigno clase ternaria
  tb_prediccion[, clase_ternaria := dfuture$clase_ternaria ]

  # asigno ganancias
  tb_prediccion[, ganancia := -0.025 ]
  tb_prediccion[clase_ternaria=="BAJA+2", ganancia := 0.975 ]

  # reordeno, y acumulada
  setorder( tb_prediccion, -prob )
  tb_prediccion[, gan_acum := cumsum(ganancia)]

  # media movil de ancho 400
  tb_prediccion[,
    gan_suavizada := frollmean(
      x= gan_acum,
      n= 400,
      align= "center",
      na.rm= TRUE,
      hasNA= TRUE
    )
  ]

  tb_predicciones[[ as.character(s) ]] <- tb_prediccion
}

Warning message in froll(fun = "mean", x = x, n = n, fill = fill, algo = algo, align = align, :
“hasNA is deprecated, use has.nf instead”
Warning message in froll(fun = "mean", x = x, n = n, fill = fill, algo = algo, align = align, :
“hasNA is deprecated, use has.nf instead”
Warning message in froll(fun = "mean", x = x, n = n, fill = fill, algo = algo, align = align, :
“hasNA is deprecated, use has.nf instead”
Warning message in froll(fun = "mean", x = x, n = n, fill = fill, algo = algo, align = align, :
“hasNA is deprecated, use has.nf instead”
Warning message in froll(fun = "mean", x = x, n = n, fill = fill, algo = algo, align = align, :
“hasNA is deprecated, use has.nf instead”
Warning message in froll(fun = "mean", x = x, n = n, fill = fill, algo = algo, align = align, :
“hasNA is deprecated, use has.nf instead”
Warning message in froll(fun = "mean", x = x, n = n, fill = fill, algo = algo, align = align, :
“hasNA is deprecated, use has.nf instead”
Warning message in froll(fun = "me

In [ ]:
tb_prediccion[, sum(!is.na(gan_suavizada) )]
tb_prediccion[, .N]

[1] 12803

[1] 13202

In [ ]:
# Este es la ganancia a reportar, una por semilla
options(digits= 8)

resultados <- list()
for( s in PARAM$semillas ) {
  tb_prediccion <- tb_predicciones[[ as.character(s) ]]

  resultado <- list()
  resultado$ganancia_suavizada_max <- max( tb_prediccion$gan_suavizada, na.rm=TRUE )
  resultado$envios <- which.max( tb_prediccion$gan_suavizada)

  resultados[[ as.character(s) ]] <- resultado
}

# tabla con el resultado de cada semilla
data.table(
  semilla= PARAM$semillas,
  ganancia_suavizada_max= sapply( resultados, function(r) r$ganancia_suavizada_max ),
  envios= sapply( resultados, function(r) r$envios )
)

semilla,ganancia_suavizada_max,envios
<dbl>,<dbl>,<int>
133811,33.6100,1019
159763,32.6975,1320
175403,31.2425,955
208309,31.6800,765
249589,32.2650,780
254039,32.1875,803
254279,33.2425,831
292793,33.6775,796
325421,31.7875,1117


In [ ]:
# grabo las ganancias, una tabla por semilla
for( s in PARAM$semillas ) {
  fwrite( tb_predicciones[[ as.character(s) ]],
    file= paste0("semilla_", s, "/ganancias.txt"),
    sep= "\t"
  )
}

In [ ]:
# genero el grafico, uno por semilla

for( s in PARAM$semillas ) {
  tb_prediccion <- tb_predicciones[[ as.character(s) ]]
  resultado <- resultados[[ as.character(s) ]]

  tb_prediccion[, envios:= .I]

  pdf( paste0("semilla_", s, "/curva_de_ganancia.pdf") )

  plot(
    x= tb_prediccion$envios,
    y= tb_prediccion$gan_acum,
    type= "l",
    col= "gray",
    xlim= c(0, 6000),
    ylim= c(0, 50),
    main= paste0("Semilla ", s, " gan prom= ", as.integer(resultado$ganancia_suavizada_max), " envios= ", resultado$envios),
    xlab= "Envios",
    ylab= "Ganancia",
    panel.first= grid()
  )

  dev.off()

  tb_predicciones[[ as.character(s) ]] <- tb_prediccion
}

In [ ]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

# parametros de la corrida
write_yaml( PARAM, file="PARAM.yml")

# parametros y resultado de cada semilla
for( s in PARAM$semillas ) {
  PARAM_s <- PARAM
  PARAM_s$semilla_final <- s
  PARAM_s$resultado <- resultados[[ as.character(s) ]]

  write_yaml( PARAM_s, file= paste0("semilla_", s, "/PARAM.yml") )
}

Loading required package: yaml



In [ ]:
# ---- salida en planillas: una fila por semilla y un resumen por corrida ----
cortes <- c(500, 750, 1000, 1250, 1500)

tb_semillas <- rbindlist( lapply( PARAM$semillas, function(s) {
  tb <- tb_predicciones[[ as.character(s) ]]
  fila <- data.table(
    experimento      = PARAM$experimento,
    variante         = PARAM$variante,
    semilla          = s,
    pesos_mes        = paste(names(PARAM$pesos_mes), PARAM$pesos_mes, sep= ":", collapse= " "),
    meses_training   = paste(sort(PARAM$trainingstrategy$training), collapse = " "),
    meses_final      = paste(sort(PARAM$trainingstrategy$final_train), collapse = " "),
    validate         = paste(PARAM$trainingstrategy$validate, collapse = " "),
    future           = paste(PARAM$trainingstrategy$future, collapse = " "),
    AUC_valid        = PARAM$out$lgbm$AUC,
    num_leaves       = PARAM$out$lgbm$mejores_hiperparametros$num_leaves,
    min_data_in_leaf = PARAM$out$lgbm$mejores_hiperparametros$min_data_in_leaf,
    num_iterations   = PARAM$out$lgbm$mejores_hiperparametros$num_iterations,
    ganancia_max     = resultados[[ as.character(s) ]]$ganancia_suavizada_max,
    envios_max       = resultados[[ as.character(s) ]]$envios
  )
  for( e in cortes ) set( fila, j= paste0("gan_", e), value= tb$gan_suavizada[e] )
  fila
}) )

tb_resumen <- tb_semillas[, list(
  experimento     = PARAM$experimento,
  variante        = PARAM$variante,
  n_semillas      = .N,
  gan_media       = mean(ganancia_max),
  gan_desvio      = sd(ganancia_max),
  gan_min         = min(ganancia_max),
  gan_max         = max(ganancia_max),
  envios_media    = mean(envios_max),
  gan_500_media   = mean(gan_500),
  gan_750_media   = mean(gan_750),
  gan_1000_media  = mean(gan_1000),
  gan_1250_media  = mean(gan_1250),
  gan_1500_media  = mean(gan_1500),
  minutos_corrida = round(as.numeric(difftime(Sys.time(), t0, units= "mins")), 1)
)]

archivo_semillas <- "/content/buckets/b1/exp/resultados_final_exp07_B_semillas.csv"
archivo_resumen  <- "/content/buckets/b1/exp/resumen_final_exp07_B_corridas.csv"

fwrite( tb_semillas, archivo_semillas, append= file.exists(archivo_semillas) )
fwrite( tb_resumen,  archivo_resumen,  append= file.exists(archivo_resumen) )

tb_semillas
tb_resumen

experimento,variante,semilla,pesos_mes,meses_training,meses_final,validate,future,AUC_valid,num_leaves,min_data_in_leaf,num_iterations,ganancia_max,envios_max,gan_500,gan_750,gan_1000,gan_1250,gan_1500
<dbl>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
9270,F2_W7_escalones_descendentes_1_0.75_0.5_0.25,133811,202005:1 202006:1 202007:1 202008:0.75 202009:0.75 202010:0.75 202011:0.5 202012:0.5 202101:0.5 202102:0.25 202103:0.25 202104:0.25,202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104,202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104 202105,202105,202107,0.95220092,256,64,366,33.6100,1019,27.5575,31.6450,33.5625,31.7000,28.4075
9270,F2_W7_escalones_descendentes_1_0.75_0.5_0.25,159763,202005:1 202006:1 202007:1 202008:0.75 202009:0.75 202010:0.75 202011:0.5 202012:0.5 202101:0.5 202102:0.25 202103:0.25 202104:0.25,202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104,202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104 202105,202105,202107,0.95220092,256,64,366,32.6975,1320,28.8475,30.4700,30.8125,32.3400,31.4800
9270,F2_W7_escalones_descendentes_1_0.75_0.5_0.25,175403,202005:1 202006:1 202007:1 202008:0.75 202009:0.75 202010:0.75 202011:0.5 202012:0.5 202101:0.5 202102:0.25 202103:0.25 202104:0.25,202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104,202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104 202105,202105,202107,0.95220092,256,64,366,31.2425,955,26.7350,30.6675,31.1200,31.1550,29.8025
9270,F2_W7_escalones_descendentes_1_0.75_0.5_0.25,208309,202005:1 202006:1 202007:1 202008:0.75 202009:0.75 202010:0.75 202011:0.5 202012:0.5 202101:0.5 202102:0.25 202103:0.25 202104:0.25,202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104,202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104 202105,202105,202107,0.95220092,256,64,366,31.6800,765,27.5925,31.6100,31.1250,29.8350,27.9400
9270,F2_W7_escalones_descendentes_1_0.75_0.5_0.25,249589,202005:1 202006:1 202007:1 202008:0.75 202009:0.75 202010:0.75 202011:0.5 202012:0.5 202101:0.5 202102:0.25 202103:0.25 202104:0.25,202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104,202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104 202105,202105,202107,0.95220092,256,64,366,32.2650,780,28.6150,32.1625,31.3825,28.6775,29.6950
9270,F2_W7_escalones_descendentes_1_0.75_0.5_0.25,254039,202005:1 202006:1 202007:1 202008:0.75 202009:0.75 202010:0.75 202011:0.5 202012:0.5 202101:0.5 202102:0.25 202103:0.25 202104:0.25,202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104,202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104 202105,202105,202107,0.95220092,256,64,366,32.1875,803,28.9700,32.0425,31.0325,30.1900,28.8200
9270,F2_W7_escalones_descendentes_1_0.75_0.5_0.25,254279,202005:1 202006:1 202007:1 202008:0.75 202009:0.75 202010:0.75 202011:0.5 202012:0.5 202101:0.5 202102:0.25 202103:0.25 202104:0.25,202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104,202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104 202105,202105,202107,0.95220092,256,64,366,33.2425,831,28.2775,32.8000,32.8275,30.5900,27.4050
9270,F2_W7_escalones_descendentes_1_0.75_0.5_0.25,292793,202005:1 202006:1 202007:1 202008:0.75 202009:0.75 202010:0.75 202011:0.5 202012:0.5 202101:0.5 202102:0.25 202103:0.25 202104:0.25,202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104,202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104 202105,202105,202107,0.95220092,256,64,366,33.6775,796,29.2850,33.5375,32.6875,30.5475,29.0300
9270,F2_W7_escalones_descendentes_1_0.75_0.5_0.25,325421,202005:1 

experimento,variante,n_semillas,gan_media,gan_desvio,gan_min,gan_max,envios_media,gan_500_media,gan_750_media,gan_1000_media,gan_1250_media,gan_1500_media,minutos_corrida
<dbl>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
9270,F2_W7_escalones_descendentes_1_0.75_0.5_0.25,30,32.504917,0.99781323,30.69,34.6875,971.36667,28.463583,31.44375,31.78975,31.19925,29.7435,112.5


In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Nov 02 02:13:57 AM 2026"